In [1]:
from __future__ import annotations

import math
import os
from getpass import getpass
from typing import Any

from google import genai

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("GOOGLE_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-Goog-Studio-Api-Key": os.environ["GOOGLE_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
google_client = genai.Client(
    api_key=os.environ["GOOGLE_API_KEY"],
)

response = google_client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain in one short sentence what text embeddings are.",
)

print(response.text)

Text embeddings are numerical representations of text (like words, phrases, or documents) that capture their semantic meaning and context, allowing computers to process and understand relationships between them.


In [5]:
embedding_response = google_client.models.embed_content(
    model="gemini-embedding-2",
    contents="Weaviate is a vector database used for semantic search and RAG.",
)

embedding = embedding_response.embeddings[0].values

print("Embedding type:", type(embedding))
print("Embedding length:", len(embedding))
print("First 10 values:", embedding[:10])

Embedding type: <class 'list'>
Embedding length: 3072
First 10 values: [-0.0034960175, 0.03160315, -0.013314722, 0.005782377, -0.0041091656, -0.01765101, -0.001493584, 0.027517764, 0.014330842, -0.04013441]


In [6]:
def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot_product = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot_product / (norm_a * norm_b)

In [7]:
texts = [
    "Weaviate is useful for semantic search.",
    "Vector databases help find similar meanings.",
    "Pizza with mozzarella and tomatoes is delicious.",
]

embeddings = []

for text in texts:
    result = google_client.models.embed_content(
        model="gemini-embedding-2",
        contents=text,
    )

    embeddings.append(result.embeddings[0].values)

print("similarity 0-1:", cosine_similarity(embeddings[0], embeddings[1]))
print("similarity 0-2:", cosine_similarity(embeddings[0], embeddings[2]))
print("similarity 1-2:", cosine_similarity(embeddings[1], embeddings[2]))

similarity 0-1: 0.8258868887236132
similarity 0-2: 0.3851109981302002
similarity 1-2: 0.39264009704747016


In [8]:
GOOGLE_EMBEDDING_MODEL = "gemini-embedding-2"

print("Google embedding model:", GOOGLE_EMBEDDING_MODEL)

Google embedding model: gemini-embedding-2


In [9]:
documents = [
    {
        "title": "Weaviate Vector Search",
        "content": (
            "Weaviate stores objects and vectors. It can search by semantic similarity "
            "using embeddings generated from text."
        ),
        "topic": "vector_database",
    },
    {
        "title": "Hybrid Search",
        "content": (
            "Hybrid search combines keyword search with vector search. It is useful when "
            "users mix exact technical terms with natural language."
        ),
        "topic": "search",
    },
    {
        "title": "RAG Application",
        "content": (
            "Retrieval-Augmented Generation retrieves relevant context from a vector database "
            "and sends it to a generative model."
        ),
        "topic": "rag",
    },
    {
        "title": "FastAPI Backend",
        "content": (
            "FastAPI is a Python web framework for building APIs with type hints, async support "
            "and automatic OpenAPI documentation."
        ),
        "topic": "backend",
    },
    {
        "title": "Docker Containers",
        "content": (
            "Docker containers package applications with dependencies and help run services "
            "in isolated environments."
        ),
        "topic": "devops",
    },
]

In [13]:
COLLECTION_NAME = "GoogleGeminiEmbeddingTest"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

google_docs = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_google_gemini(
        name="content_vector",
        source_properties=[
            "title",
            "content",
        ],
        model=GOOGLE_EMBEDDING_MODEL,
    ),
    generative_config=wvc.config.Configure.Generative.google_gemini(
        model="gemini-2.5-flash",
        temperature=0.2,
        max_output_tokens=512,
    ),
    properties=[
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="topic",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: GoogleGeminiEmbeddingTest


In [12]:
import inspect

print("text2vec_google_gemini:")
print(inspect.signature(wvc.config.Configure.Vectors.text2vec_google_gemini))

print("\ngenerative google_gemini exists:")
print(hasattr(wvc.config.Configure.Generative, "google_gemini"))

if hasattr(wvc.config.Configure.Generative, "google_gemini"):
    print("\ngoogle_gemini:")
    print(inspect.signature(wvc.config.Configure.Generative.google_gemini))

text2vec_google_gemini:
(*, name: Optional[str] = None, quantizer: Optional[weaviate.collections.classes.config_base._QuantizerConfigCreate] = None, dimensions: Optional[int] = None, model: Optional[str] = None, title_property: Optional[str] = None, task_type: Optional[str] = None, source_properties: Optional[List[str]] = None, vector_index_config: Optional[weaviate.collections.classes.config_vector_index._VectorIndexConfigCreate] = None, vectorize_collection_name: bool = True) -> weaviate.collections.classes.config_vectors._VectorConfigCreate

generative google_gemini exists:
True

google_gemini:
(max_output_tokens: Optional[int] = None, model: Optional[str] = None, temperature: Optional[float] = None, top_k: Optional[int] = None, top_p: Optional[float] = None) -> weaviate.collections.classes.config._GenerativeProvider


In [14]:
result = google_docs.data.insert_many(documents)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [15]:
response = google_docs.aggregate.over_all(total_count=True)

print("Total objects:", response.total_count)

Total objects: 5


In [16]:
response = google_docs.query.near_text(
    query="How can I build semantic search with vectors?",
    target_vector="content_vector",
    limit=3,
    return_properties=[
        "title",
        "content",
        "topic",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Distance: 0.249484121799469
Title: Weaviate Vector Search
Topic: vector_database
Content: Weaviate stores objects and vectors. It can search by semantic similarity using embeddings generated from text.
--------------------------------------------------------------------------------
Distance: 0.27270352840423584
Title: RAG Application
Topic: rag
Content: Retrieval-Augmented Generation retrieves relevant context from a vector database and sends it to a generative model.
--------------------------------------------------------------------------------
Distance: 0.31360161304473877
Title: Hybrid Search
Topic: search
Content: Hybrid search combines keyword search with vector search. It is useful when users mix exact technical terms with natural language.
--------------------------------------------------------------------------------


In [17]:
response = google_docs.query.near_text(
    query="retrieving context for a language model answer",
    target_vector="content_vector",
    limit=3,
    return_properties=[
        "title",
        "content",
        "topic",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Distance: 0.2809255123138428
Title: RAG Application
Topic: rag
Content: Retrieval-Augmented Generation retrieves relevant context from a vector database and sends it to a generative model.
--------------------------------------------------------------------------------
Distance: 0.35875725746154785
Title: Weaviate Vector Search
Topic: vector_database
Content: Weaviate stores objects and vectors. It can search by semantic similarity using embeddings generated from text.
--------------------------------------------------------------------------------
Distance: 0.37002086639404297
Title: Hybrid Search
Topic: search
Content: Hybrid search combines keyword search with vector search. It is useful when users mix exact technical terms with natural language.
--------------------------------------------------------------------------------


In [18]:
response = google_docs.generate.near_text(
    query="What should I learn to build a RAG application?",
    target_vector="content_vector",
    grouped_task=(
        "Answer using only the retrieved documents. "
        "Explain which concepts are useful for building a RAG application. "
        "Keep the answer concise."
    ),
    limit=4,
    return_properties=[
        "title",
        "content",
        "topic",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print("-", obj.properties["title"], "|", obj.properties["topic"])

The useful concepts for building a RAG application are:

*   **Retrieval-Augmented Generation (RAG)**: This involves retrieving relevant context from a vector database and sending it to a generative model.
*   **Vector Database**: Stores objects and vectors, enabling search by semantic similarity using embeddings generated from text (e.g., Weaviate).
*   **Generative Model**: The model that receives the retrieved context to generate a response.
*   **Hybrid Search**: Combines keyword search with vector search, useful for retrieval when users mix exact technical terms with natural language.

Sources:
- RAG Application | rag
- FastAPI Backend | backend
- Weaviate Vector Search | vector_database
- Hybrid Search | search


In [19]:
response = google_docs.generate.near_text(
    query="What should I learn to build a RAG application?",
    target_vector="content_vector",
    grouped_task=(
        "Answer using only the retrieved documents. "
        "Explain which concepts are useful for building a RAG application. "
        "Keep the answer concise."
    ),
    limit=4,
    return_properties=[
        "title",
        "content",
        "topic",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print("-", obj.properties["title"], "|", obj.properties["topic"])

The useful concepts for building a RAG application are:

*   **Retrieval-Augmented Generation (RAG)**: The core concept itself, involving retrieving relevant context from a vector database and sending it to a generative model.
*   **Weaviate Vector Search**: A vector database that stores objects and vectors, enabling search by semantic similarity using embeddings, which is crucial for the retrieval step in RAG.
*   **Hybrid Search**: Combines keyword search with vector search, useful for improving retrieval when users mix exact terms with natural language.

Sources:
- RAG Application | rag
- FastAPI Backend | backend
- Weaviate Vector Search | vector_database
- Hybrid Search | search


In [20]:
def get_json(
    url: str,
    *,
    params: dict[str, Any] | None = None,
) -> dict[str, Any]:
    headers = {
        "User-Agent": "PythonPracticeNotebook/1.0 (learning project)",
        "Accept": "application/json",
    }

    with httpx.Client(
        timeout=30.0,
        follow_redirects=True,
        headers=headers,
    ) as http_client:
        response = http_client.get(url, params=params)
        response.raise_for_status()
        return response.json()

In [47]:
def fetch_wikipedia_summary(title: str) -> dict[str, Any]:
    url = "https://en.wikipedia.org/w/api.php"

    data = get_json(
        url,
        params={
            "action": "query",
            "format": "json",
            "formatversion": "2",
            "prop": "extracts",
            "exintro": "true",
            "explaintext": "true",
            "titles": title,
            "redirects": "1",
            "origin": "*",
        },
    )

    pages = data.get("query", {}).get("pages", [])

    if not pages:
        raise RuntimeError(f"No Wikipedia page returned for: {title}")

    page = pages[0]

    if page.get("missing"):
        raise RuntimeError(f"Wikipedia page missing for: {title}")

    return {
        "title": page.get("title", title),
        "description": "travel destination",
        "extract": page.get("extract", ""),
    }

In [48]:
def fetch_destination_weather(
    *,
    latitude: float,
    longitude: float,
    forecast_days: int = 5,
) -> dict[str, Any]:
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": [
            "weather_code",
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_probability_max",
            "wind_speed_10m_max",
        ],
        "timezone": "auto",
        "forecast_days": forecast_days,
    }

    return get_json(url, params=params)

In [49]:
def fetch_country_info(country_name: str) -> dict[str, Any]:
    url = f"https://restcountries.com/v3.1/name/{country_name}"

    data = get_json(
        url,
        params={
            "fullText": "true",
        },
    )

    if not data:
        raise RuntimeError(f"No country data returned for {country_name}")

    return data[0]

In [50]:
WEATHER_CODE_MAP = {
    0: "clear sky",
    1: "mainly clear",
    2: "partly cloudy",
    3: "overcast",
    45: "fog",
    48: "depositing rime fog",
    51: "light drizzle",
    53: "moderate drizzle",
    55: "dense drizzle",
    61: "slight rain",
    63: "moderate rain",
    65: "heavy rain",
    71: "slight snow fall",
    73: "moderate snow fall",
    75: "heavy snow fall",
    80: "slight rain showers",
    81: "moderate rain showers",
    82: "violent rain showers",
    95: "thunderstorm",
}


def describe_weather_code(code: int) -> str:
    return WEATHER_CODE_MAP.get(code, f"unknown weather code {code}")

In [51]:
def normalize_weather_forecast(weather_response: dict[str, Any]) -> dict[str, Any]:
    daily = weather_response["daily"]

    temperatures_max = daily["temperature_2m_max"]
    temperatures_min = daily["temperature_2m_min"]
    rain_probabilities = daily["precipitation_probability_max"]
    wind_speeds = daily["wind_speed_10m_max"]
    weather_codes = daily["weather_code"]

    avg_max_temp = sum(temperatures_max) / len(temperatures_max)
    avg_min_temp = sum(temperatures_min) / len(temperatures_min)
    avg_rain_probability = sum(rain_probabilities) / len(rain_probabilities)
    max_wind_speed = max(wind_speeds)

    most_common_weather_code = max(
        set(weather_codes),
        key=weather_codes.count,
    )

    weather_description = describe_weather_code(int(most_common_weather_code))

    if avg_rain_probability >= 60:
        travel_weather_label = "rain risk"
    elif max_wind_speed >= 40:
        travel_weather_label = "windy"
    elif avg_max_temp >= 28:
        travel_weather_label = "hot"
    elif avg_max_temp >= 18:
        travel_weather_label = "pleasant"
    elif avg_max_temp >= 10:
        travel_weather_label = "mild"
    else:
        travel_weather_label = "cold"

    weather_summary = (
        f"Forecast summary: average maximum temperature {avg_max_temp:.1f}°C, "
        f"average minimum temperature {avg_min_temp:.1f}°C, average rain probability "
        f"{avg_rain_probability:.1f}%, maximum wind speed {max_wind_speed:.1f} km/h. "
        f"Typical weather: {weather_description}. Travel weather label: {travel_weather_label}."
    )

    return {
        "avg_max_temp_c": round(avg_max_temp, 2),
        "avg_min_temp_c": round(avg_min_temp, 2),
        "avg_rain_probability": round(avg_rain_probability, 2),
        "max_wind_speed_kmh": round(max_wind_speed, 2),
        "weather_description": weather_description,
        "travel_weather_label": travel_weather_label,
        "weather_summary": weather_summary,
    }

In [52]:
def normalize_country(country_response: dict[str, Any]) -> dict[str, Any]:
    name = country_response["name"]["common"]
    region = country_response.get("region", "unknown")
    subregion = country_response.get("subregion", "unknown")
    population = int(country_response.get("population", 0))

    capital_values = country_response.get("capital", [])
    capital = capital_values[0] if capital_values else "unknown"

    currencies = country_response.get("currencies", {})
    currency_codes = list(currencies.keys())
    currency = currency_codes[0] if currency_codes else "unknown"

    languages = country_response.get("languages", {})
    language_values = list(languages.values())
    main_language = language_values[0] if language_values else "unknown"

    return {
        "country_region": region,
        "country_subregion": subregion,
        "country_population": population,
        "country_capital": capital,
        "country_currency": currency,
        "country_language": main_language,
        "country_summary": (
            f"{name} is in {region}, {subregion}. Capital: {capital}. "
            f"Population: {population}. Currency: {currency}. Main language: {main_language}."
        ),
    }

In [53]:
def normalize_wikipedia_summary(wiki_response: dict[str, Any]) -> dict[str, Any]:
    title = wiki_response.get("title", "unknown")
    description = wiki_response.get("description", "unknown")
    extract = wiki_response.get("extract", "")

    return {
        "wiki_title": title,
        "wiki_description": description,
        "wiki_extract": extract,
        "wiki_summary": f"{title}: {description}. {extract}",
    }

In [54]:
def build_destination_profile(destination: dict[str, Any]) -> dict[str, Any]:
    print("Fetching:", destination["city"], destination["country"])

    wiki_response = fetch_wikipedia_summary(destination["wiki_title"])

    weather_response = fetch_destination_weather(
        latitude=destination["latitude"],
        longitude=destination["longitude"],
        forecast_days=5,
    )

    country_response = fetch_country_info(destination["country"])

    wiki_data = normalize_wikipedia_summary(wiki_response)
    weather_data = normalize_weather_forecast(weather_response)
    country_data = normalize_country(country_response)

    combined_summary = (
        f"Travel destination profile for {destination['city']}, {destination['country']}. "
        f"Travel style: {destination['travel_style']}. "
        f"{wiki_data['wiki_summary']} "
        f"{weather_data['weather_summary']} "
        f"{country_data['country_summary']}"
    )

    return {
        "city": destination["city"],
        "country": destination["country"],
        "latitude": float(destination["latitude"]),
        "longitude": float(destination["longitude"]),
        "travel_style": destination["travel_style"],
        **wiki_data,
        **weather_data,
        **country_data,
        "combined_summary": combined_summary,
    }

In [55]:
destinations = [
    {
        "city": "Barcelona",
        "country": "Spain",
        "wiki_title": "Barcelona",
        "latitude": 41.3874,
        "longitude": 2.1686,
        "travel_style": "city break, architecture, beach, food",
    },
    {
        "city": "Rome",
        "country": "Italy",
        "wiki_title": "Rome",
        "latitude": 41.9028,
        "longitude": 12.4964,
        "travel_style": "history, architecture, food, city break",
    },
    {
        "city": "Athens",
        "country": "Greece",
        "wiki_title": "Athens",
        "latitude": 37.9838,
        "longitude": 23.7275,
        "travel_style": "history, ancient ruins, warm weather, food",
    },
    {
        "city": "Lisbon",
        "country": "Portugal",
        "wiki_title": "Lisbon",
        "latitude": 38.7223,
        "longitude": -9.1393,
        "travel_style": "city break, ocean views, food, walking",
    },
    {
        "city": "Dubrovnik",
        "country": "Croatia",
        "wiki_title": "Dubrovnik",
        "latitude": 42.6507,
        "longitude": 18.0944,
        "travel_style": "coast, old town, sea, sightseeing",
    },
    {
        "city": "Prague",
        "country": "Czechia",
        "wiki_title": "Prague",
        "latitude": 50.0755,
        "longitude": 14.4378,
        "travel_style": "city break, architecture, walking, history",
    },
]

print("Destinations:", len(destinations))

Destinations: 6


In [56]:
destination_records = []

for destination in destinations:
    try:
        record = build_destination_profile(destination)
        destination_records.append(record)
    except Exception as error:
        print("Failed:", destination)
        print(error)

print("Destination records:", len(destination_records))

Fetching: Barcelona Spain
Failed: {'city': 'Barcelona', 'country': 'Spain', 'wiki_title': 'Barcelona', 'latitude': 41.3874, 'longitude': 2.1686, 'travel_style': 'city break, architecture, beach, food'}
Client error '403 Forbidden' for url 'https://en.wikipedia.org/w/api.php?action=query&format=json&formatversion=2&prop=extracts&exintro=true&explaintext=true&titles=Barcelona&redirects=1&origin=%2A'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403
Fetching: Rome Italy
Failed: {'city': 'Rome', 'country': 'Italy', 'wiki_title': 'Rome', 'latitude': 41.9028, 'longitude': 12.4964, 'travel_style': 'history, architecture, food, city break'}
Client error '403 Forbidden' for url 'https://en.wikipedia.org/w/api.php?action=query&format=json&formatversion=2&prop=extracts&exintro=true&explaintext=true&titles=Rome&redirects=1&origin=%2A'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403
Fetching: Athens Greece
Failed: {'cit

In [39]:
for record in destination_records:
    print(record["city"], record["country"])
    print("Weather:", record["travel_weather_label"])
    print("Avg max temp:", record["avg_max_temp_c"])
    print("Rain probability:", record["avg_rain_probability"])
    print("Currency:", record["country_currency"])
    print("Wiki:", record["wiki_description"])
    print("Summary:", record["combined_summary"][:1000])
    print("-" * 100)

In [57]:
COLLECTION_NAME = "GoogleTravelDestinationProfile"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

travel_profiles = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_google_gemini(
        name="destination_vector",
        source_properties=[
            "city",
            "country",
            "travel_style",
            "wiki_summary",
            "weather_summary",
            "country_summary",
            "combined_summary",
        ],
        model=GOOGLE_EMBEDDING_MODEL,
    ),
    generative_config=wvc.config.Configure.Generative.google_gemini(
        model="gemini-2.5-flash",
        temperature=0.2,
        max_output_tokens=700,
    ),
    properties=[
        wvc.config.Property(name="city", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="latitude", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="longitude", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="travel_style", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="wiki_title", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="wiki_description", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="wiki_extract", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="wiki_summary", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="avg_max_temp_c", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="avg_min_temp_c", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="avg_rain_probability", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="max_wind_speed_kmh", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="weather_description", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="travel_weather_label", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="weather_summary", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_region", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_subregion", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_population", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="country_capital", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_currency", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_language", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_summary", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="combined_summary", data_type=wvc.config.DataType.TEXT),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: GoogleTravelDestinationProfile


In [58]:
COLLECTION_NAME = "GoogleTravelDestinationProfile"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

travel_profiles = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_google_gemini(
        name="destination_vector",
        source_properties=[
            "city",
            "country",
            "travel_style",
            "wiki_summary",
            "weather_summary",
            "country_summary",
            "combined_summary",
        ],
        model=GOOGLE_EMBEDDING_MODEL,
    ),
    generative_config=wvc.config.Configure.Generative.google_gemini(
        model="gemini-2.5-flash",
        temperature=0.2,
        max_output_tokens=700,
    ),
    properties=[
        wvc.config.Property(name="city", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="latitude", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="longitude", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="travel_style", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="wiki_title", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="wiki_description", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="wiki_extract", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="wiki_summary", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="avg_max_temp_c", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="avg_min_temp_c", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="avg_rain_probability", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="max_wind_speed_kmh", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="weather_description", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="travel_weather_label", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="weather_summary", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_region", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_subregion", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_population", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="country_capital", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_currency", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_language", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_summary", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="combined_summary", data_type=wvc.config.DataType.TEXT),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: GoogleTravelDestinationProfile


In [37]:
import httpx

In [59]:
for destination in destinations:
    print("Testing:", destination["city"], destination["country"])

    try:
        wiki_response = fetch_wikipedia_summary(destination["wiki_title"])
        print("Wikipedia OK")
    except Exception as error:
        print("Wikipedia FAILED:", repr(error))

    try:
        weather_response = fetch_destination_weather(
            latitude=destination["latitude"],
            longitude=destination["longitude"],
            forecast_days=5,
        )
        print("Weather OK")
    except Exception as error:
        print("Weather FAILED:", repr(error))

    try:
        country_response = fetch_country_info(destination["country"])
        print("Country OK")
    except Exception as error:
        print("Country FAILED:", repr(error))

    print("-" * 100)

Testing: Barcelona Spain
Wikipedia FAILED: HTTPStatusError("Client error '403 Forbidden' for url 'https://en.wikipedia.org/w/api.php?action=query&format=json&formatversion=2&prop=extracts&exintro=true&explaintext=true&titles=Barcelona&redirects=1&origin=%2A'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403")
Weather OK
Country OK
----------------------------------------------------------------------------------------------------
Testing: Rome Italy
Wikipedia FAILED: HTTPStatusError("Client error '403 Forbidden' for url 'https://en.wikipedia.org/w/api.php?action=query&format=json&formatversion=2&prop=extracts&exintro=true&explaintext=true&titles=Rome&redirects=1&origin=%2A'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403")
Weather OK
Country OK
----------------------------------------------------------------------------------------------------
Testing: Athens Greece
Wikipedia FAILED: HTTPStatusError("Clie

In [60]:
collections_to_delete = [
    "GoogleGeminiEmbeddingTest",
    "GoogleTravelDestinationProfile",
]

for collection_name in collections_to_delete:
    if client.collections.exists(collection_name):
        client.collections.delete(collection_name)
        print("Deleted:", collection_name)
    else:
        print("Not found:", collection_name)

client.close()

print("Client closed.")

Deleted: GoogleGeminiEmbeddingTest
Deleted: GoogleTravelDestinationProfile
Client closed.
